# 数据增强：让模型看到更多变化
> 难度标签：高级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：09_计算机视觉 | 源文件：数据增强.py | 核心功能：torchvision.transforms、随机变换、增强策略

## 概述

数据增强通过对训练图像施加随机变换（翻转、旋转、裁剪、颜色抖动等），人工扩大训练集规模，提升模型泛化能力。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

## 数学原理

### 1. 数据增强的正则化原理

数据增强通过扩充训练集来减少过拟合，等价于在损失函数中引入隐式正则化：

$$\mathcal{L}_{aug} = \frac{1}{|D|}\sum_{x \in D} \mathbb{E}_{T \sim p(T)}[L(f(T(x)), y)]$$

其中 $T$ 是随机变换，$p(T)$ 是变换的分布。增强增加了训练样本的有效数量。

### 2. 几何变换

**随机裁剪**（RandomCrop）：从图像中随机裁剪子区域

$$T_{crop}(I) = I[x_0:x_0+h, \; y_0:y_0+w], \quad x_0 \sim \text{Uniform}(0, \text{pad})$$

padding 先在边缘填充像素再裁剪，保证输出尺寸不变。

**随机翻转**（RandomHorizontalFlip）：

$$T_{flip}(I)_{i,j} = I_{i, W-1-j}, \quad \text{概率 } p = 0.5$$

**随机旋转**（RandomRotation）：

$$\begin{pmatrix} x' \\ y' \end{pmatrix} = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix} \begin{pmatrix} x \\ y \end{pmatrix}, \quad \theta \sim \text{Uniform}(-15°, 15°)$$

### 3. 颜色变换

**颜色抖动**（ColorJitter）：随机调整亮度、对比度、饱和度

$$I'_{c} = \alpha \cdot I_c + \beta, \quad \alpha \sim [1-b, 1+b], \; \beta \sim [-b, b]$$

**标准化**（Normalize）：减去均值，除以标准差

$$\hat{I}_c = \frac{I_c - \mu_c}{\sigma_c}$$

MNIST 的 $\mu = 0.1307, \sigma = 0.3081$（全数据集统计量）。

### 4. 随机擦除（Random Erasing）

在图像上随机遮挡矩形区域，强迫模型关注全局特征：

$$I'_{i,j} = \begin{cases} r & (i,j) \in R_{erase} \\ I_{i,j} & \text{otherwise} \end{cases}$$

其中 $R_{erase}$ 是随机矩形区域，$r \sim \text{Uniform}(0, 1)$ 是随机填充值。

### 5. 仿射变换

综合旋转、平移、缩放、剪切的线性变换：

$$\begin{pmatrix} x' \\ y' \end{pmatrix} = \begin{pmatrix} a & b \\ c & d \end{pmatrix} \begin{pmatrix} x \\ y \end{pmatrix} + \begin{pmatrix} t_x \\ t_y \end{pmatrix}$$

`RandomAffine(degrees=10, translate=(0.1, 0.1))` 控制旋转范围和平移范围。

### 6. Mixup 数据增强

将两个样本线性插值：

$$\tilde{x} = \lambda x_i + (1-\lambda) x_j, \quad \tilde{y} = \lambda y_i + (1-\lambda) y_j, \quad \lambda \sim \text{Beta}(\alpha, \alpha)$$

### 7. 数据增强的效果

数据增强通过增加训练集多样性来降低模型方差：

$$\text{Var}[\hat{f}] \propto \frac{1}{n_{eff}}, \quad n_{eff} = n \times |\text{augmentations}|$$

代码中对比有增强 vs 无增强的测试准确率差异，验证增强的正则化效果。

### 8. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `RandomCrop(28, padding=4)` | 随机裁剪 + padding，平移增强 |
| `RandomHorizontalFlip(p=0.5)` | 以 0.5 概率水平翻转 |
| `RandomRotation(15)` | $\theta \sim U(-15°, 15°)$ |
| `ColorJitter(0.2, 0.2)` | 亮度/对比度随机扰动 |
| `RandomErasing(p=0.5)` | 随机遮挡矩形区域 |
| `Normalize((0.1307,), (0.3081,))` | $\hat{x} = (x - \mu) / \sigma$ |
| `transforms.Compose([...])` | 变换链 $T = T_n \circ \cdots \circ T_1$ |

### 增强技术详解

运行 增强技术详解 的代码，观察算法在该环节的行为。

In [ ]:

def demonstrate_transforms():
    """展示各种数据增强操作的参数和含义"""
    print("--- 常用数据增强技术 ---\n")

    augmentation_info = [
        ("RandomCrop(28, padding=4)",
         "随机裁剪：先在图像四周填充 4 像素，再随机裁剪回 28×28。\n"
         "    作用：模拟物体在图像中不同位置，提高平移不变性。"),
        ("RandomHorizontalFlip(p=0.5)",
# --- 继续 ---
         "随机水平翻转：以 50% 概率左右镜像。\n"
         "    作用：增加水平方向的样本多样性。"),
        ("RandomRotation(15)",
         "随机旋转：在 [-15°, +15°] 范围内随机旋转。\n"
         "    作用：模拟不同角度拍摄，提高旋转鲁棒性。"),
# --- 继续 ---
        ("ColorJitter(brightness=0.2, contrast=0.2)",
         "颜色抖动：随机调整亮度和对比度。\n"
         "    作用：模拟不同光照条件（注：MNIST 是灰度图，效果有限）。"),
        ("RandomAffine(degrees=10, translate=(0.1, 0.1))",
         "随机仿射变换：在指定角度和平移范围内随机变换。\n"
# --- 继续 ---
         "    作用：综合模拟旋转和平移的组合变换。"),
        ("RandomErasing(p=0.5, scale=(0.02, 0.15))",
         "随机擦除：在图像上随机遮挡一小块区域。\n"
         "    作用：强迫模型关注全局特征，避免过度依赖局部区域。"),
        ("Normalize((0.1307,), (0.3081,))",
# --- 继续 ---
         "标准化：将像素值减去均值除以标准差。\n"
         "    作用：使数据分布与预训练模型匹配，加速收敛。"),
    ]

    for name, desc in augmentation_info:
        print(f"  {name}")
        print(f"    {desc}")
        print()

### 增强流水线构建

运行 增强流水线构建 的代码，观察算法在该环节的行为。

In [ ]:

def get_transforms(use_augmentation=True):
    """
    构建数据预处理流水线。

    参数:
        use_augmentation: 是否启用数据增强

    返回:
        训练变换、测试变换
    """
    # 测试集不做增强，只做必要的预处理
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ])

    if use_augmentation:
        # 训练集：增强 + 预处理
        train_transform = transforms.Compose([
            transforms.RandomCrop(28, padding=4),        # 随机裁剪
            transforms.RandomHorizontalFlip(p=0.5),      # 随机水平翻转
            transforms.RandomRotation(15),                # 随机旋转
            transforms.ToTensor(),                        # 转为张量
# --- 继续 ---
            transforms.Normalize((0.1307,), (0.3081,)),  # 标准化
            transforms.RandomErasing(p=0.3),              # 随机擦除（需在张量上操作）
        ])
    else:
        # 训练集：不做增强
        train_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),
        ])

    return train_transform, test_transform

### 简单分类模型

在分类任务上训练模型并评估性能。

In [ ]:

class SimpleCNN(nn.Module):
    """用于 MNIST 分类的简单 CNN"""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
# --- 继续 ---
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

### 训练和评估

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:

def train_epoch(model, loader, criterion, optimizer, device):
    """训练一个 epoch"""
    model.train()
    total_loss, correct, total = 0, 0, 0
    for data, target in loader:
# --- 继续 ---
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
# --- 继续 ---
        optimizer.step()
        total_loss += loss.item() * data.size(0)
        correct += (output.argmax(1) == target).sum().item()
        total += data.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, device):
    """评估模型准确率"""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
# --- 循环处理 ---
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            correct += (output.argmax(1) == target).sum().item()
            total += target.size(0)
# --- 返回结果 ---
    return correct / total

### 对比实验

对比不同模型或配置的性能差异。

In [ ]:

def run_comparison():
    """对比有增强和无增强的训练效果"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    batch_size = 128
    epochs = 3
# --- 赋值/配置 ---
    lr = 1e-3

    results = {}

    for aug_flag, aug_name in [(False, "无增强"), (True, "有增强")]:
        print(f"\n{'='*40}")
        print(f"实验: {aug_name}")
        print(f"{'='*40}")

        train_transform, test_transform = get_transforms(use_augmentation=aug_flag)

        train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=train_transform)
        test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=test_transform)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

        model = SimpleCNN().to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        for epoch in range(1, epochs + 1):
            train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
            test_acc = evaluate(model, test_loader, device)
            print(f"  Epoch {epoch}: 训练损失={train_loss:.4f}, "
                  f"训练准确率={train_acc*100:.2f}%, 测试准确率={test_acc*100:.2f}%")

        results[aug_name] = test_acc

    print(f"\n{'='*40}")
    print("对比结果:")
    for name, acc in results.items():
        print(f"  {name}: 测试准确率 {acc*100:.2f}%")
    print("注: 数据增强在更大数据集和更多 epoch 下效果更显著")

### 主流程

运行 主流程 的代码，观察算法在该环节的行为。

In [ ]:

def main():
    print("=" * 50)
    print("数据增强 — 计算机视觉中的数据扩充")
    print("=" * 50)

    # 展示各种增强技术
    demonstrate_transforms()

    # 打印增强流水线
    train_transform, test_transform = get_transforms(use_augmentation=True)
    print("--- 训练增强流水线 ---")
    print(f"  {train_transform}\n")
    print("--- 测试预处理流水线 ---")
    print(f"  {test_transform}\n")

    # 对比实验
    run_comparison()

    # 补充说明
    print("\n--- 数据增强最佳实践 ---")
    print("  1. 增强应与任务匹配：医学图像不适合水平翻转（左右不对称）")
    print("  2. 增强强度要适中：过度增强会引入噪声，降低数据质量")
    print("  3. 测试时不做增强：评估应使用标准预处理，确保可复现")
    print("  4. 增强与正则化结合：配合 Dropout、权重衰减效果更好")
# --- 输出结果 ---
    print("  5. 高级方法：AutoAugment、RandAugment 可自动搜索最优增强策略")


if __name__ == "__main__":
    main()

## 关键代码解释

```python
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2),
    transforms.RandomResizedCrop(224),
    transforms.ToTensor()
])
```

## 注意事项

1. 增强策略需要符合业务场景（医学图像不能随便翻转）
2. 测试集不做增强
3. AutoAugment 可以自动搜索最优策略

## 总结与延伸

以上代码展示了 **数据增强** 的完整流程。

**解读要点**：
- 关注 **损失函数** 的收敛曲线
- 观察训练/验证损失是否出现分叉（过拟合信号）
- 对比不同超参数（学习率、层数）的影响

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **Mixup/CutMix**：混合两张图像的增强
- **RandAugment**：简化的自动增强
- **文本/音频增强**：不同领域的增强方法
